# 样本均值的抽样分布

> 对应代码：`MathStatsCode/code_in_notes/Chap.6/statistic_mean.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：从 N(1,4) 中重复抽样10000次（每次样本量100），用直方图展示样本均值的抽样分布（近似正态）。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.6`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.6")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

这是“**样本均值的抽样分布**”的完整模拟。流程分四步：①`clear all` 清空；②`set seed 19880505` 固定种子；③`frame create means m` 新建一个名为 means 的数据框（frame），其中只有一个变量 m，用来存放每次模拟的样本均值；④`forvalues i=1/10000` 循环10000次：每轮 `clear` 清空当前数据、`set obs 100` 生成100个样本、`x=rnormal(1,2)` 从总体 N(1,4)（均值1、标准差2）抽样、`su x` 求均值、`frame post means (r(mean))` 把均值写入 means 框。
`frame` 是 Stata 16+ 的数据框功能：模拟结果可以单独存放而不影响工作数据；`frame post` 像“填表”一样追加一行，是模拟循环的标准套路。

模拟结束后，`frame change means` 切换到存放结果的数据框；`su m` 查看10000个样本均值的均值与标准差（后者即标准误的模拟值，理论上 σ/√n=2/10=0.2）；`hist m` 画均值直方图——可以看到其形状近似正态、以总体均值1为中心，这正是**中心极限定理**的体现（原总体是正态分布时精确成立）；`graph export` 导出图片。


In [ ]:
%%stata
clear all
set seed 19880505
frame create means m
forvalues i=1/10000{
    clear
    qui{
        set obs 100
        gen x=rnormal(1,2)
        su x
        frame post means (r(mean))
    }
}
frame change means
su m
hist m
